In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')

2.12.0+cu126
12.6
True
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [1]:
import os, re, sys
sys.path.insert(0, "..")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import glob
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login

from candidates.bm25_retriever import BM25Retriever
from utils.prompt_builder import build_prompt_zero_shot, build_prompt_mcq
from utils.response_parser import parse_response
from utils.metrics import is_correct, recall_at_k, compute_metrics, print_metrics


c:\Lobna\Télécom Paris\2A\Projet SD\ProjectDSAI2026_13\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
MODE = "mcq_bm25"
MODEL_NAME = "Qwen/Qwen3-0.6B"
K = 10
MAX_EXAMPLES = 200      
BATCH_SIZE = 8

TITLES_PATH = "../data/wikidata_qid_titles.csv"
INDEX_PATH = "../candidates/index/bm25.pkl"
BENCHMARKS = "../data/benchmarks"
RESULTS_DIR = "../evaluation/results"
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

## ELGOLD Dataset

Files `.txt` with annotations `{{mention|TYPE|gold_title}}`.
We extract each mention + context (150 chars from each side).

In [7]:
ANNOT_RE = re.compile(r"\{\{([^|{}]+)\|([^|{}]+)\|([^{}]+)\}\}")

def parse_elgold_file(path, context_window=150):
    raw = Path(path).read_text(encoding="utf-8")
    clean = ANNOT_RE.sub(lambda m: m.group(1).strip(), raw)
    examples, cursor, last_end = [], 0, 0
    for match in ANNOT_RE.finditer(raw):
        mention = match.group(1).strip()
        ent_type = match.group(2).strip()
        gold = match.group(3).strip()
        cursor += len(raw[last_end:match.start()])
        end = cursor + len(mention)
        snippet = clean[max(0, cursor-context_window):min(len(clean), end+context_window)]
        examples.append({"file": Path(path).name, "mention": mention,
                          "entity_type": ent_type, "gold": gold, "context": snippet})
        cursor, last_end = end, match.end()
    return examples

rows = []
for f in glob.glob(f"{BENCHMARKS}/*.txt"):
    rows.extend(parse_elgold_file(f))

df = pd.DataFrame(rows).drop_duplicates()
if MAX_EXAMPLES:
    df = df.head(MAX_EXAMPLES)

print(f"Exemples : {len(df)}")
print(f"Fichiers : {df['file'].nunique()}")
print(f"Types : {df['entity_type'].value_counts().to_dict()}")
df.head(3)

Exemples : 200
Fichiers : 6
Types : {'GPE': 81, 'PERSON': 42, 'ORG': 40, 'WORK_OF_ART': 11, 'PRODUCT': 10, 'NORP': 6, 'LOC': 5, 'EVENT': 4, 'FAC': 1}


,file,mention,entity_type,gold,context
0,1_1.txt,Robbie Coltrane,PERSON,Robbie Coltrane,"Robbie Coltrane, a long-time Scottish-born com..."
1,1_1.txt,Scottish,NORP,Scottish people,"Robbie Coltrane, a long-time Scottish-born com..."
2,1_1.txt,British,NORP,British people,"Robbie Coltrane, a long-time Scottish-born com..."


## BM25 index

The index is built the first time (only once) from `wikidata_qid_titles.csv` and saved as `.pkl`.  
The following times, we load it from the `.pkl`  

The index contains Wikipedia titles (candidate entities).  

In [8]:
if Path(INDEX_PATH).exists():
    print("Index found → loading...")
    retriever = BM25Retriever.load(INDEX_PATH)
else:
    print("Index not found → construction...")
    titles_df = pd.read_csv(TITLES_PATH)
    print(f"Columns : {titles_df.columns.tolist()}")

    titles = titles_df["title"].dropna().tolist()
    print(f"{len(titles):,} titles loaded")
    retriever = BM25Retriever.from_entity_list(titles)
    retriever.save(INDEX_PATH)

print(f"Index ready : {len(retriever.entity_titles):,} entities")

Index found → loading...
[BM25] Index loaded from ../candidates/index/bm25.pkl (1201852 entities)
Index ready : 1,201,852 entities


BM25 verification (small test)

In [9]:
for _, row in df.head(5).iterrows():
    candidates = retriever.retrieve(row["mention"], k=K)
    gold_found = "✓" if recall_at_k(row["gold"], candidates) else "✗"
    print(f"{gold_found} mention='{row['mention']}'  gold='{row['gold']}'")
    print(f"  → {candidates}\n")

✓ mention='Robbie Coltrane'  gold='Robbie Coltrane'
  → ['Robbie Coltrane', 'John Coltrane', '5893 Coltrane', 'Alice Coltrane', 'Chi Coltrane', 'Standard Coltrane', 'Coltrane (disambiguation)', 'Coltrane Jazz', 'Ravi Coltrane', 'Robbie Grabarz']

✓ mention='Scottish'  gold='Scottish people'
  → ['Scottish', 'Scottish Gaelic', 'Scottish Government', 'Scottish Deerhound', 'Scottish Terrier', 'Scottish Fold', 'Scottish Highlands', 'Scottish people', 'Scottish Parliament', 'Scottish Borders']

✗ mention='British'  gold='British people'
  → ['British', 'British Columbia', 'British Museum', 'British English', 'British Empire', 'British Airways', 'British Library', 'British Longhair', 'British Shorthair', 'British Isles']

✗ mention='Cracker'  gold='Cracker (British TV series)'
  → ['Cracker', 'Cracker (computing)', 'Oyster cracker', 'Cracker Jack', 'Cracker (band)', 'TUC (cracker)', 'Cracker (food)', 'Christmas cracker', 'Christmas Cracker (film)', 'Saltine cracker challenge']

✗ mention='Ha

## Recall@K on benchmark

Recall@K = pourcentage d'exemples où le gold est dans les K candidats BM25.  

L'accuracy ne peut pas le dépasser.

In [10]:
import sys
print(sys.executable)

import accelerate
print(accelerate.__version__)

c:\Lobna\Télécom Paris\2A\Projet SD\ProjectDSAI2026_13\.venv\Scripts\python.exe
1.13.0


In [11]:
# Generate candidates for all the benchmark
df["candidates"] = df["mention"].apply(lambda m: retriever.retrieve(m, k=K))
df["gold_in_candidates"] = df.apply(
    lambda row: recall_at_k(row["gold"], row["candidates"]), axis=1
)

recall = df["gold_in_candidates"].mean()
print(f"Recall@{K} (BM25) : {recall:.2%}")

missed = df[~df["gold_in_candidates"]]
print(f"\nGold absent des candidats : {len(missed)}/{len(df)} exemples")
print("Exemples ratés par BM25 :")
missed[["mention", "gold", "candidates"]].head(5)

Recall@10 (BM25) : 62.00%

Gold absent des candidats : 76/200 exemples
Exemples ratés par BM25 :


,mention,gold,candidates
2,British,British people,"[British, British Columbia, British Museum, Br..."
3,Cracker,Cracker (British TV series),"[Cracker, Cracker (computing), Oyster cracker,..."
4,Harry Potter,Harry Potter (film series),"[Harry Potter, Harry Potter (disambiguation), ..."
5,WME,Endeavor (company),"[WME, WME Group, Universe, Earth, Human, Canad..."
7,Press Association,PA Media,"[Amateur press association, Hollywood Foreign ..."


## Loading model

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
).eval()
print(f"Loaded on : {next(model.parameters()).device}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
c:\Lobna\Télécom Paris\2A\Projet SD\ProjectDSAI2026_13\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|████

Loaded on : cuda:0


In [13]:
def generate_batch(prompts, max_new_tokens=20):
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=1024
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    return [
        tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        for out in outputs
    ]

## Test on examples

In [14]:
for rec in df.head(3).to_dict("records"):
    prompt = (
        build_prompt_zero_shot(rec["context"], rec["mention"], rec["entity_type"])
        if MODE == "zero_shot"
        else build_prompt_mcq(rec["context"], rec["mention"], rec["entity_type"], rec["candidates"])
    )
    raw = generate_batch([prompt])[0]
    parsed = parse_response(raw, mode="zero_shot" if MODE == "zero_shot" else "mcq",
                            candidates=rec["candidates"])
    ok = is_correct(parsed["predicted"], rec["gold"])
    print(f"{'✓' if ok else '✗'}  mention='{rec['mention']}'")
    print(f"   gold : {rec['gold']}")
    print(f"   raw output: {repr(raw)}")
    print(f"   predicted : {parsed['predicted']}")
    print()

✓  mention='Robbie Coltrane'
   gold : Robbie Coltrane
   raw output: "A\n\nNow, let's see. The text says Robbie Coltrane, a long-time Scottish"
   predicted : Robbie Coltrane

✗  mention='Scottish'
   gold : Scottish people
   raw output: 'A\n```\n```\n```\n```\n```\n```\n```\n```\n```'
   predicted : Scottish

✗  mention='British'
   gold : British people
   raw output: 'A\n```\n```\n```\n```\n```\n```\n```\n```\n```'
   predicted : British



## Zero-shot Evaluation

In [ ]:
results  = []
records  = df.to_dict("records")
llm_mode = "zero_shot" if MODE == "zero_shot" else "mcq"

for i in tqdm(range(0, len(records), BATCH_SIZE)):
    batch = records[i : i + BATCH_SIZE]
    prompts = [
        build_prompt_zero_shot(r["context"], r["mention"], r["entity_type"])
        if MODE == "zero_shot"
        else build_prompt_mcq(r["context"], r["mention"], r["entity_type"], r["candidates"])
        for r in batch
    ]
    try:
        raws = generate_batch(prompts)
    except Exception as e:
        for r in batch:
            results.append({**r, "predicted": "", "correct": False, "raw_output": "", "error": str(e)})
        continue

    for rec, raw in zip(batch, raws):
        try:
            parsed = parse_response(raw, mode=llm_mode, candidates=rec["candidates"])
            pred, error = parsed["predicted"], ""
        except Exception as e:
            pred, error = "", str(e)

        results.append({
            "file": rec["file"],
            "mention": rec["mention"],
            "entity_type": rec["entity_type"],
            "gold": rec["gold"],
            "candidates": rec["candidates"],
            "gold_in_candidates":rec["gold_in_candidates"],
            "predicted": pred,
            "correct": is_correct(pred, rec["gold"]) if not error else False,
            "raw_output": raw,
            "error": error,
        })

df_results = pd.DataFrame(results)
slug = MODEL_NAME.replace("/", "_")
out_path = f"{RESULTS_DIR}/{MODE}_{slug}.csv"
df_results.to_csv(out_path, index=False)
print(f"Saved : {out_path}")

## Results

In [ ]:
metrics = compute_metrics(df_results)
print_metrics(metrics, model_name=f"{MODE} / {MODEL_NAME}")

if MODE == "mcq_bm25":
    print(f"\nRecall@{K} BM25 (plafond) : {recall:.2%}")
    print(f"Gap plafond → accuracy : -{(recall - metrics['accuracy']):.2%}  (LLM errors)")

# Errors: when gold was among candidates but the LLM did not select it
ok_retrieval = df_results[df_results["gold_in_candidates"]] if "gold_in_candidates" in df_results.columns else df_results
llm_errors = ok_retrieval[~ok_retrieval["correct"] & (ok_retrieval["error"] == "")]
print(f"\nErreurs LLM (gold était dans les candidats) : {len(llm_errors)}")
for _, r in llm_errors.head(5).iterrows():
    print(f"  '{r['mention']}' | gold='{r['gold']}' | pred='{r['predicted']}' | raw={repr(r['raw_output'])}")